In [2]:
import ee
from datetime import datetime, timezone

# 初始化（若未认证，取消下面两行的注释）
ee.Authenticate()
ee.Initialize(project='where-has-sar')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [3]:
#对于保护区里每一个要素（feature）而言，遍历，然后输出每个保护区这个时间段存在sar图像的时间信息

# ========== 1. 加载保护区 FeatureCollection ==========
asset_path = 'projects/where-has-sar/assets/Navigator_AllSites_2604_final_keep'
fc = ee.FeatureCollection(asset_path)

#asset_path是 GEE 云端资产的路径（Asset ID）
#作用：读取已经上传到 GEE 云端 Assets 仓库里的数据，用`ee.FeatureCollection(asset_path)` / `ee.Image(asset_path)`加载。

total = fc.size().getInfo()


#fc.size() GEE 云端算子，向 GEE 服务器发起请求：统计这个 FeatureCollection 里面一共有多少个 Feature
#返回值类型：`ee.Number`（这还是**GEE 云端对象，不是 Python 数字**！）
#.getInfo()把云端`ee.Number` → Python 整数`int`


print(f"共加载 {total} 个保护区")

共加载 5069 个保护区


In [5]:
#增加scenid
# ========== 2. 定义时间范围：2015年 ==========
start_date = ee.Date('2015-01-01')
end_date   = ee.Date('2016-01-01')

# 为每个保护区生成一条记录
def make_row(feature):
    feature = ee.Feature(feature)
    region  = feature.geometry()

    s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
          .filterBounds(region)
          .filterDate(start_date, end_date)
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
          .select('VV')
          .sort('system:time_start'))   # 排序，保证 dates 与 scene_ids 一一对应

    timestamps = s1.aggregate_array('system:time_start')
    scene_ids  = s1.aggregate_array('system:index')   # ← 新增：场景 ID 列表

    dates      = timestamps.map(lambda ts: ee.Date(ts).format('YYYY-MM-dd'))
#为保证 dates、scene_ids、coverages 的长度严格相等（都是一景一个），把.distinct()被删去
    date_str   = dates.join(';')
    scene_str  = scene_ids.join(';')                  # ← 新增：用分号拼成字符串

    return ee.Feature(None, {
        'site_id':     feature.get('SITE_ID'),
        'site_name':   feature.get('NAME'),
        'image_count': s1.size(),
        'dates':       date_str,
        'scene_ids':   scene_str,        # ← 新增列
        'has_image':   s1.size().gt(0)
    })

export_fc = fc.map(make_row)

# ========== 3. 导出 CSV 到 Google Drive ==========
task = ee.batch.Export.table.toDrive(
    collection=export_fc,
    description='sar_2015_protected_areas_with_sceneid',
    folder='GEE_exports',
    fileNamePrefix='sar_2015_protected_areas_with_sceneid',
    fileFormat='CSV',
    selectors=['site_id', 'site_name', 'image_count', 'dates', 'scene_ids', 'has_image']
)
task.start()
print("✅ 导出任务已提交，请到 https://code.earthengine.google.com/tasks 查看进度")

✅ 导出任务已提交，请到 https://code.earthengine.google.com/tasks 查看进度


In [6]:
#每个小buffer跑索引代码

# ========== 1. 加载 6x6km buffer 图层 ==========
buffers = ee.FeatureCollection(
    'projects/where-has-sar/assets/MPA_grids_region_test1'   # ← 改成你的 buffer 资产路径
)

# ========== 2. 时间范围 ==========
start_date = ee.Date('2015-01-01')
end_date   = ee.Date('2016-01-01')

# ========== 3. 每个 buffer 生成一行（包含覆盖率计算）==========
def make_buffer_row(feature):
    feat = ee.Feature(feature)
    geom = feat.geometry()

    # 计算 buffer 面积（分母）。如果是精确的 6*6km，可以直接用 36e6  这里先直接用
    buf_area = ee.Number(36e6) 
    #buf_area = geom.area(ee.ErrorMargin(1))

    # 搜索该 buffer 命中的 SAR 影像
    s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
          .filterBounds(geom)
          .filterDate(start_date, end_date)
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
          .select('VV')
          .sort('system:time_start'))     # 必须排序，保证 dates/scene_ids/coverages 三者顺序一致

    # 给每景影像挂一个 coverage 属性 = 相交面积 / buffer 面积
    def add_coverage(img):
        img = ee.Image(img)
        # 计算影像几何与 buffer 几何的交集
        inter = img.geometry().intersection(geom, ee.ErrorMargin(1))
        # 计算覆盖率 (0~1 之间)
        cov = inter.area(ee.ErrorMargin(1)).divide(buf_area)
        return img.set('coverage', cov)

    s1 = s1.map(add_coverage)

    # 提取日期（去重）
    dates = (s1.aggregate_array('system:time_start')
               .map(lambda t: ee.Date(t).format('YYYY-MM-dd'))
               .distinct()
               .join(';'))

    # 提取 scene_id（不去重，一景一个）
    scene_ids = s1.aggregate_array('system:index').join(';')

    # 提取覆盖率，保留 4 位小数
    coverages = (s1.aggregate_array('coverage')
                   .map(lambda c: ee.Number(c).format('%.4f'))
                   .join(';'))

    return ee.Feature(None, {
        'FID':         feat.get('FID'),
        'SITE_ID':     feat.get('SITE_ID'),
        'site_name':   feat.get('site_name'),
        'image_count': s1.size(),
        'dates':       dates,
        'scene_ids':   scene_ids,
        'coverages':   coverages,
    })

# ========== 4. 分块提交导出任务 ==========
# 获取所有唯一的 SITE_ID
site_ids = buffers.aggregate_array('SITE_ID').distinct().getInfo()
print(f"共加载 {len(site_ids)} 个保护区（SITE_ID）")

# 每批处理 50 个 SITE_ID（大约 2万个 buffer，可根据实际大小调整）
BATCH_SIZE = 50
batches = [site_ids[i:i + BATCH_SIZE] for i in range(0, len(site_ids), BATCH_SIZE)]

print(f"共需提交 {len(batches)} 个导出任务...")

for idx, batch_ids in enumerate(batches):
    # 筛选出当前批次的 buffer
    sub_buffers = buffers.filter(ee.Filter.inList('SITE_ID', batch_ids))
    
    # 映射处理函数
    out_fc = sub_buffers.map(make_buffer_row)
    
    # 提交任务
    task = ee.batch.Export.table.toDrive(
        collection=out_fc,
        description=f'sar_buffer_batch_{idx:03d}',
        folder='GEE_exports',
        fileNamePrefix=f'sar_buffer_batch_{idx:03d}',
        fileFormat='CSV',
        selectors=['FID', 'SITE_ID', 'site_name', 'image_count', 'dates', 'scene_ids', 'coverages']
    )
    task.start()
    print(f"✅ 任务 {idx+1}/{len(batches)} 已提交 (SITE_ID: {batch_ids[0]}...{batch_ids[-1]})")

print("🎉 所有任务已提交！请到 https://code.earthengine.google.com/tasks 查看进度。")


共加载 2 个保护区（SITE_ID）
共需提交 1 个导出任务...


✅ 任务 1/1 已提交 (SITE_ID: AICHN138...AIEST302)
🎉 所有任务已提交！请到 https://code.earthengine.google.com/tasks 查看进度。
